## 실습 문제

### 문제 1 — 나만의 오디오북 만들기
좋아하는 문장이나 짧은 글을 골라 `audio.speech`(`gpt-4o-mini-tts`)로 음성을 생성하세요.
- `voice` 를 바꿔보고, `instructions` 로 어조(예: "차분한 다큐멘터리 내레이션")를 지정해 보세요.
- 생성한 mp3를 `Audio` 로 재생해 확인하세요.

### 문제 2 — 받아쓰기 검증
문제 1에서 만든 음성을 `audio.transcriptions`(`gpt-4o-transcribe`)로 다시 텍스트로 변환하고,
원본 텍스트와 얼마나 일치하는지 비교해 보세요.

### 테스트 입력 예시
- 텍스트: `"인공지능은 인간의 가능성을 확장하는 도구입니다."`
- instructions: `"느리고 또박또박, 강연자처럼 말하세요."`

## 정답

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
client = OpenAI()

### 문제 1 — TTS로 오디오북 만들기

In [2]:
from pathlib import Path
from IPython.display import Audio, display

original_text = "인공지능은 인간의 가능성을 확장하는 도구입니다."
answer_file = Path("answer_audiobook.mp3")

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="nova",
    input=original_text,
    instructions="느리고 또박또박, 강연자처럼 말하세요.",
) as response:
    response.stream_to_file(answer_file)

print("저장 완료:", answer_file)
display(Audio(str(answer_file)))

저장 완료: answer_audiobook.mp3


### 문제 2 — STT로 받아쓰기 후 원본과 비교

In [3]:
# 음성 -> 텍스트
with open("answer_audiobook.mp3", "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file,
    )

recognized = transcript.text.strip()
print("원본 텍스트  :", original_text)
print("받아쓰기 결과:", recognized)

원본 텍스트  : 인공지능은 인간의 가능성을 확장하는 도구입니다.
받아쓰기 결과: 인공지능은 인간의 가능성을 확장하는 도구입니다.


In [4]:
# 간단한 일치도 비교 (문자 단위 유사도)
from difflib import SequenceMatcher

ratio = SequenceMatcher(None, original_text, recognized).ratio()
print(f"일치도: {ratio * 100:.1f}%")

if ratio >= 0.9:
    print("거의 완벽하게 일치합니다!")
elif ratio >= 0.7:
    print("대체로 잘 인식되었습니다. 구두점/띄어쓰기 차이일 수 있어요.")
else:
    print("차이가 큽니다. 음성 품질이나 모델을 확인해 보세요.")

일치도: 100.0%
거의 완벽하게 일치합니다!
